# 03. 분석 테이블 단위와 분모 연결 지도

## 분석 개요

- **목적**: 세 분석 테이블의 한 행 의미와 포함 범위, 하류 질문을 연결한다.
- **분모**: 각 분석 테이블의 전체 행 수이며 분석 테이블마다 포함 범위가 다르다.
- **관측 기간**: 원본 이벤트 로그의 2019-10-01부터 2020-02-29까지다.
- **제외 범위**: 퍼널 전환율과 대표 첫 구매·실험 지표는 계산하지 않는다.

---
## 1. 목적과 위치

02에서 생성한 세 분석 테이블을 같은 지표의 대체본이 아니라 서로 다른 질문을 위한 분석 기반으로 정리한다. 03은 행 수와 정의만 연결하고, 실제 퍼널은 04, 대표 첫 구매 경로와 실험 설계는 05에서 계산한다.

In [1]:
import os
import re
import sys
from pathlib import Path
from urllib.parse import quote_plus

import pandas as pd
from dotenv import load_dotenv
from sqlalchemy import create_engine

def find_project_root():
    for base in (Path.cwd(), *Path.cwd().parents):
        if (base / 'cache_context.json').is_file() and (base / 'sql').is_dir():
            return base
    raise FileNotFoundError(f'프로젝트 루트 없음 (cwd={Path.cwd()})')

PROJECT_ROOT = find_project_root()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from query_cache import QueryCache, load_queries

load_dotenv(PROJECT_ROOT / '.env')
engine = create_engine(
    f"mysql+pymysql://{os.getenv('DB_USER')}:{quote_plus(os.getenv('DB_PASSWORD'))}"
    f"@{os.getenv('DB_HOST')}:{os.getenv('DB_PORT')}/{os.getenv('DB_NAME')}?charset=utf8mb4"
)
SQL_FILE = PROJECT_ROOT / 'sql' / '03_mart_eda.sql'
Q = load_queries(SQL_FILE)
if list(Q) != ['mart_inventory']:
    raise ValueError(f'03 named query 구성이 다르다: {list(Q)}')
inventory_sql = re.sub(r'(?m)^\s*--.*(?:\n|$)', '', Q['mart_inventory']).lstrip()
if re.match(r'^(\w+)', inventory_sql).group(1).upper() != 'SELECT':
    raise ValueError('mart_inventory는 SELECT 문이어야 한다.')

QUERY_CACHE = QueryCache(
    engine=engine,
    sql_file=SQL_FILE,
    upstream_sql_files=(PROJECT_ROOT / 'sql' / '02_preprocessing_mart.sql',),
)

---
## 2. 세 분석 테이블 구성 현황

각 분석 테이블의 전체 행 수를 DB에서 직접 계산한다. 이전 실행 기준값은 조회 결과를 대체하지 않고 현재 분석 테이블이 02 기본 점검과 같은지만 확인한다.

In [2]:
inventory = QUERY_CACHE.run('mart_inventory')
inventory['실제_행수'] = inventory['실제_행수'].astype('int64')

EXPECTED_ROWS = {
    'mart_session': 4_499_479,
    'mart_session_product': 12_102_048,
    'mart_user_product_session': 13_385_787,
}
if len(inventory) != 3 or inventory['마트'].duplicated().any():
    raise AssertionError('mart_inventory는 중복 없는 세 마트 3행이어야 한다.')
actual_rows = inventory.set_index('마트')['실제_행수'].to_dict()
if actual_rows != EXPECTED_ROWS:
    raise AssertionError(f'02 smoke check와 행 수가 다르다: {actual_rows}')

[mart_inventory] cache HIT 5e70190b19e2 · 3행


---
## 3. 분석 단위와 하류 사용처

`mart_session_product`와 `mart_user_product_session`은 모두 세션·상품 단위지만 포함 이벤트와 보존 컬럼이 다르다. 아래 표의 행 수는 각 분석 테이블 자체의 분모이며 서로 직접 비교하는 전환율이 아니다.

In [3]:
mart_definitions = pd.DataFrame([
    {
        '마트': 'mart_session',
        '한 행의 의미': '유효 세션 1개',
        '포함 범위': 'user_session이 있고 단일 사용자이며 1일 이하인 세션',
        '사용 노트북': '04',
        '답하는 질문': '한 세션 안에서 행동이 어떤 순서로 이어지는가',
    },
    {
        '마트': 'mart_session_product',
        '한 행의 의미': '유효 세션 × 상품 1개',
        '포함 범위': '조회·담기·구매 중 하나가 있는 세션·상품; remove-only 제외',
        '사용 노트북': '04',
        '답하는 질문': '같은 세션·같은 상품의 조회→담기→구매가 이어지는가',
    },
    {
        '마트': 'mart_user_product_session',
        '한 행의 의미': '유효 세션 × 상품 1개',
        '포함 범위': 'remove-only 포함; 구매 여정용 행동 최초·최종 시각 보존',
        '사용 노트북': '05',
        '답하는 질문': '세션을 연결했을 때 사용자·상품별 대표 첫 구매 경로는 무엇인가',
    },
])
analysis_unit_map = mart_definitions.merge(inventory, on='마트', how='left', validate='one_to_one')
analysis_unit_map = analysis_unit_map[[
    '마트', '한 행의 의미', '실제_행수', '포함 범위', '사용 노트북', '답하는 질문',
]]
analysis_unit_map

,마트,한 행의 의미,실제_행수,포함 범위,사용 노트북,답하는 질문
0,mart_session,유효 세션 1개,4499479,user_session이 있고 단일 사용자이며 1일 이하인 세션,04,한 세션 안에서 행동이 어떤 순서로 이어지는가
1,mart_session_product,유효 세션 × 상품 1개,12102048,조회·담기·구매 중 하나가 있는 세션·상품; remove-only 제외,04,같은 세션·같은 상품의 조회→담기→구매가 이어지는가
2,mart_user_product_session,유효 세션 × 상품 1개,13385787,remove-only 포함; 구매 여정용 행동 최초·최종 시각 보존,05,세션을 연결했을 때 사용자·상품별 대표 첫 구매 경로는 무엇인가


---
## 4. 분모 해석 원칙

> ### 섹션 결론
>
> - 세 분석 테이블은 같은 원본 이벤트 로그 데이터에서 만들어졌지만 답하려는 질문과 행의 포함 범위가 다르다.
> - `mart_session_product`는 remove-only 조합을 제외하지만 `mart_user_product_session`은 구매 여정 복원을 위해 이를 포함한다.
> - 따라서 서로 다른 분석 테이블의 행 수나 비율을 직접 나누거나 전환율 차이로 해석하지 않는다.
> - 각 비율은 해당 분석에서 선언한 분석 테이블·cohort의 분모 안에서만 해석한다.

---
## 5. 04·05로 인계

- 04는 `mart_session`으로 세션 단위 퍼널을, `mart_session_product`로 같은 세션·같은 상품 퍼널을 계산한다. 두 퍼널도 분모와 상품 동일성 조건이 달라 직접 비교하지 않는다.
- 05는 `mart_user_product_session`의 실제 이벤트 시각으로 세션을 연결하고, 사용자·상품 단위 30일 cohort와 대표 첫 구매 경로를 구성한다.
- 세션 퍼널 전환율, 대표 첫 구매 경로, 30일 구매 전환율, 장바구니 구매율, 실험 기준선은 이 노트북에서 반복 계산하지 않는다.